# AURA DIAMOND 30K — Drive-Persistent + Auto-Resume
Saves checkpoints to Google Drive. Auto-resumes if runtime disconnects.
Just rerun all cells — it picks up where it left off.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/SotoAlt/aura.git /content/aura 2>/dev/null || (cd /content/aura && git pull)
!pip install -q -r /content/aura/world_model/requirements-diamond.txt librosa
!mkdir -p /content/video_data/_raw/frames
!wget -q -O /content/video_data/_raw/video.mp4 https://0x0.st/Pp_q.mp4
!ffmpeg -y -i /content/video_data/_raw/video.mp4 -ar 22050 -ac 1 /content/video_data/_raw/audio.wav 2>/dev/null
!ffmpeg -y -i /content/video_data/_raw/video.mp4 -vf "crop=min(iw\,ih):min(iw\,ih),scale=64:64:flags=lanczos" -r 10 /content/video_data/_raw/frames/frame_%05d.png 2>/dev/null
!cd /content/aura && python -c "from pathlib import Path; from world_model.data.video import build_episodes; build_episodes(Path('/content/video_data/_raw/frames'), Path('/content/video_data/_raw/audio.wav'), '/content/video_data')"

In [ ]:
import os
ckpt = '/content/drive/MyDrive/aura/checkpoints/diamond_30k.ckpt'
os.makedirs(os.path.dirname(ckpt), exist_ok=True)
resume_flag = f'--resume {ckpt}' if os.path.exists(ckpt) else ''
if resume_flag:
    print(f'RESUMING from existing checkpoint: {ckpt}')
else:
    print('Starting fresh training')
!cd /content/aura && python -m world_model.diamond.train \
    --config aura_diamond \
    --data /content/video_data \
    --steps 30000 \
    --checkpoint {ckpt} \
    {resume_flag} \
    --device auto \
    --log-every 500 \
    --save-every 5000

In [ ]:
!cd /content/aura && python -m world_model.diamond.eval \
    --checkpoint /content/drive/MyDrive/aura/checkpoints/diamond_30k.ckpt \
    --data /content/video_data \
    --output /content/drive/MyDrive/aura/eval_30k

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for g in sorted(Path('/content/drive/MyDrive/aura/eval_30k').glob('*.gif')):
    print(f'\n--- {g.stem} ---')
    display(Image(filename=str(g)))

In [ ]:
import json
with open('/content/drive/MyDrive/aura/eval_30k/metrics.json') as f:
    m = json.load(f)
print('DIAMOND v0.3 Success Criteria:')
for k, v in m.get('diamond_criteria', {}).items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")